# RackSmart: Pick-Location Re-slotting Algorithm (York DC)

## Purpose
RackSmart reads a warehouse's historical pick tours and recommends which pick locations should exchange products so that pickers walk less. A swap is recommended only if the walking it saves is worth more than the labor required to move the stock.

## Method
Each planning run applies the following steps in order.

1. **Selection (who moves): marginal detour.** For each location, the marginal detour is the reduction in tour length that would result if that stop were removed from the tours that visit it, summed over the planning period. Locations are ranked from largest to smallest.
2. **Placement (where it goes): co-pick distance.** Two locations are co-picked when the same tour visits both. For a location selected to move, candidate destinations are quieter locations that have been visited. Each candidate is scored by the count-weighted walking distance to the selected location's most frequent co-pick partners (top 20 partners, ten best candidates evaluated). The lowest score is chosen.
3. **Cost test.** The affected tours are replayed with the swap, re-routed by cheapest insertion on the travel network. The swap is kept only if walking saved ($) is at least cases moved x $0.17. Selection and placement are run in two passes.
4. **SKU clusters (optional input).** Products in the same cluster are added to a location's co-pick partners. Clusters influence placement only, not selection.
5. **Aisle fence.** Partial-pallet aisles (GG, HH) never exchange products with other aisles.
6. **Stability setting.** If fewer than 60% of the busiest 7% of locations remain among the busiest from one planning period to the next, each swap must pay in every planning period. This is an interim control until forecast-driven planning is available.
7. **Backtest guard.** The plan is rebuilt from the older periods and replayed on the newest planning period. If the projected saving, scaled to a full month, does not exceed the cost of the moves, no moves are recommended.
8. **Swap limit.** `MAX_SWAPS` (default 100) retains the first N swaps accepted. Swaps are evaluated in order of marginal detour and each has already passed the cost test. Set to `None` for no limit.

## Evaluation procedure
A plan is built from the previous month's tours and replayed on the following month's tours, which the planner has not seen. Net monthly benefit is walking saved (priced at $45.50 per hour at 1.25 mph) minus the cost of moving stock ($0.17 per case, charged in full to that month). Each month is evaluated as a first re-slot from the current layout. RackSmart is compared with: no change; classic ABC (busiest locations moved nearest dispatch, no cost test); ABC with a cost test; and family grouping with a cost test.

## Required input files
Set the paths in the settings cell: the location-to-stop crosswalk, the stop-to-stop distance table (feet), the pick transaction export, the inventory-on-hand export, and the SKU cluster file (optional; set `CLUSTERS = None` to run without it).

In [ ]:
# ---- SETTINGS: edit these -------------------------------------------------------------------
CROSSWALK = "location_stop_crosswalk.csv"                        # location -> stop crosswalk (travel network)
STOP_DIST = "stop_distance_ft.csv"                               # stop-to-stop walking distance (feet)
PICKS     = "picks_sep2025_sep2026.csv"                          # pick transactions (ORD_NO, USER_NAME, CREATE_DT, FROM_LOCATION)
INVENTORY = "York_Inventory_On_Hand.csv"                         # inventory on hand (LOCATION, ITEM, UOM, QTY)
CLUSTERS  = "sku_clusters.csv"                                   # SKU clusters (sku, cluster); set to None to run without clusters
DEMAND_SCALE = 1.0      # 1.0 = real volume; 2.0 = twice as many tours of the same shape (dollars x2, moving cost unchanged)
COST_MULT    = 1.0      # 1.0 = $0.17 per case; 2.0 = moving costs double
TEST_MONTHS  = None     # e.g. ["2026-07", "2026-08"]; None = the last 4 months
WAGE, MPH, COST_PER_CASE = 45.50, 1.25, 0.17
MAX_SWAPS    = 100      # operator limit: keep the first N accepted swaps; None = no limit

## 1. Engine: input contract, walking model and planner

In [ ]:
"""The only thing the algorithm needs from any warehouse. A new dataset = a new adapter that fills this in."""
from dataclasses import dataclass, field
import numpy as np

@dataclass
class Inputs:
    name: str
    D: np.ndarray             # (n+1) x (n+1) walking distances between pick locations; last row/col = dispatch point (depot)
    depot: int                # index of the dispatch point (= n)
    cases: list               # cases (or box-equivalents) held per location; drives move cost
    golden: list              # location ids eligible as swap partners (the golden zone)
    dollars_per_unit: float   # $ of picker time per distance unit walked
    cost_per_case: float      # $ to move one case
    tours_by_period: dict     # period label -> list of tours; a tour = list of location ids in visit order
    unit_note: str = ""
    extra: dict = field(default_factory=dict)

In [ ]:
import os
"""Generic re-slotting engine: tours + a distance table in, policies and replay results out.
Same logic as the York notebook: marginal-detour ranking, golden-zone partners, cost test, replan by cheapest insertion."""
import numpy as np
from collections import defaultdict

class World:
    def __init__(self, D, depot, cases, golden, dollars_per_unit, cost_per_case):
        self.D = [list(map(float, r)) for r in np.asarray(D)]   # nested lists: fast scalar access
        self.depot = depot
        self.cases = cases            # cases (or box-equivalents) per location, list/array
        self.golden = list(golden)    # location ids
        self.dpu = dollars_per_unit   # $ per distance unit walked
        self.cpc = cost_per_case      # $ per case moved
        self.n = len(cases)

    def tour_len(self, seq):
        if not seq: return 0.0
        D, dep = self.D, self.depot
        s = D[dep][seq[0]] + D[seq[-1]][dep]
        for a, b in zip(seq, seq[1:]): s += D[a][b]
        return s

    def replan(self, seq, m):
        """Stops at moved locations are removed and re-inserted (cheapest insertion) at their new location; others keep order."""
        if not any(l in m for l in seq): return seq
        D, dep = self.D, self.depot
        cur = [l for l in seq if l not in m]
        for l in seq:
            if l in m:
                x = m[l]
                if x in cur: continue
                best_c, best_i, prev = None, 0, dep
                for i in range(len(cur) + 1):
                    nxt = cur[i] if i < len(cur) else dep
                    c = D[prev][x] + D[x][nxt] - D[prev][nxt]
                    if best_c is None or c < best_c: best_c, best_i = c, i
                    prev = nxt
                cur = cur[:best_i] + [x] + cur[best_i:]
        return cur

    def detour_ranking(self, tours):
        D, dep = self.D, self.depot
        det, vis = defaultdict(float), defaultdict(int)
        for t in tours:
            s = [dep] + list(t) + [dep]
            for i in range(1, len(s) - 1):
                x = D[s[i-1]][s[i]] + D[s[i]][s[i+1]] - D[s[i-1]][s[i+1]]
                det[s[i]] += max(x, 0.0); vis[s[i]] += 1
        order = sorted(det, key=lambda l: -det[l])
        return order, dict(vis), dict(det)

class Planner:
    """Plans swaps on a training month of tours."""
    def __init__(self, W, train, K=5, cap=150):
        self.W, self.train, self.K, self.cap = W, train, K, cap
        self.order_detour, self.vis, self.det = W.detour_ranking(train)
        self.through = defaultdict(list)
        for i, t in enumerate(train):
            for l in set(t): self.through[l].append(i)
        self.dep_dist = [W.D[W.depot][l] for l in range(W.n)]

    def cases(self, l): return self.W.cases[l]

    def quietest_golden(self, hot, moved):
        if hot in self.gset: return []
        c = [l for l in self.W.golden if l not in moved and (os.environ.get('VISITED_ONLY','1') == '0' or l in self.vis)]
        c.sort(key=lambda l: (self.vis.get(l, 0), self.cases(l), self.dep_dist[l]))
        return c

    @property
    def gset(self):
        if not hasattr(self, "_g"): self._g = set(self.W.golden)
        return self._g

    def try_swap(self, hot, partner, moved, cur_len):
        W = self.W
        touched = set(self.through.get(hot, [])) | set(self.through.get(partner, []))
        w = {**moved, hot: partner, partner: hot}
        before = sum(cur_len[i] for i in touched)
        after_lens = {i: W.tour_len(W.replan(self.train[i], w)) for i in touched}
        saved = (before - sum(after_lens.values())) * W.dpu
        cost = W.cpc * (self.cases(hot) + self.cases(partner))
        return saved, cost, w, after_lens

    def greedy(self, order, cost_test=True, n_force=None, partners=None, moved0=None):
        """Walk down 'order'; for each hot location try K partners (from 'partners', default = quietest golden spots);
        keep the best net, and only if walking saved >= move cost. Without a cost test: swap with the first partner until n_force swaps."""
        W = self.W
        partners = partners or self.quietest_golden
        moved = dict(moved0 or {}); cur_len = [W.tour_len(W.replan(t, moved)) if moved else W.tour_len(t) for t in self.train]
        n, tcost = 0, 0.0
        limit = self.cap if cost_test else n_force
        for hot in order:
            if n >= limit: break
            if hot in moved: continue
            parts = partners(hot, moved)[: (self.K if cost_test else 1)]
            best = None
            for p in parts:
                saved, cost, w, al = self.try_swap(hot, p, moved, cur_len)
                if cost_test:
                    if saved >= cost * getattr(self, 'margin', 1.0) and (best is None or saved - cost > best[0] - best[1]): best = (saved, cost, w, al)
                else:
                    best = (saved, cost, w, al)
            if best:
                _, cost, moved, al = best
                for i, v in al.items(): cur_len[i] = v
                n += 1; tcost += cost
        return moved, n, tcost

    def cluster_step(self, moved0, loc_cluster, order):
        """Step 2: busiest cluster gets the block of locations nearest the dispatch point; swap misplaced locations into their own cluster's block."""
        vis_in = defaultdict(list)
        for l, c in loc_cluster.items(): vis_in[c].append(self.vis.get(l, 0))
        ranked = sorted(vis_in, key=lambda c: -np.mean(vis_in[c]))
        pool = sorted(loc_cluster, key=lambda l: self.dep_dist[l]); block, pos = {}, 0
        for c in ranked:
            block[c] = set(pool[pos:pos + len(vis_in[c])]); pos += len(vis_in[c])
        mis = [l for l in order if l in loc_cluster and l not in block[loc_cluster[l]]]
        def own_block_spots(hot, moved):
            c = loc_cluster[hot]
            return sorted((l for l in block[c] if loc_cluster.get(l) != c and l not in moved), key=lambda l: (self.vis.get(l, 0), self.cases(l)))
        return self.greedy(mis, True, partners=own_block_spots, moved0=moved0)

    def plan(self, policy, n_ref=None):
        if policy == "detour+cost":
            return self.greedy(self.order_detour, True)
        if policy == "velocity+cost":
            order = sorted(self.vis, key=lambda l: -self.vis[l])
            return self.greedy(order, True)
        if policy == "classic_abc":
            order = sorted(self.vis, key=lambda l: -self.vis[l])
            return self.greedy(order, False, n_force=n_ref)
        raise ValueError(policy)

def replay(W, test, moved):
    """Distance saved on the test month's real tours under the new layout (units), and share saved."""
    tot_before = sum(W.tour_len(t) for t in test)
    if not moved: return 0.0, tot_before, 0.0
    saved = 0.0
    for t in test:
        if any(l in moved for l in t):
            saved += W.tour_len(t) - W.tour_len(W.replan(t, moved))
    return saved, tot_before, saved / tot_before if tot_before else 0.0

def move_cost(W, moved):
    seen, c = set(), 0.0
    for a, b in moved.items():
        if a in seen: continue
        seen.add(a); seen.add(b); c += W.cpc * (W.cases[a] + W.cases[b])
    return c

In [ ]:
"""Planner extensions used by RackSmart: the same swap rule and cost test, with additional search steps (each one a plain rule):
 1. Wider partner search (K partners instead of 5).
 2. Second ring: a hot location may swap with ANY quieter location that is closer to the dispatch point, ranked by (distance gained x traffic difference).
 3. Repeat passes until nothing new passes the money test.
 4. Cluster step: two-way swaps first (both products land in their own cluster's band), then one-way."""
from collections import defaultdict
import numpy as np
import re

REPEAT_MIN = 0.60   # top-7% busiest locations repeating less often than this between periods = unstable demand

class PlannerPlus(Planner):
    """part_sizes: how the training tours split into periods (months, or two halves of one month). With stable=True a swap's saving
    is the smaller of (total saved) and (number of periods x the worst single period's saving): it must pay in every period."""
    def __init__(self, W, train, part_sizes=None, stable=False, **kw):
        super().__init__(W, train, **kw)
        self.repeat = None
        if stable == "auto" and part_sizes and len(part_sizes) > 1:
            starts = np.cumsum([0] + list(part_sizes)); sets = []
            for a, b in zip(starts[:-1], starts[1:]):
                c = defaultdict(int)
                for t in train[a:b]:
                    for l in set(t): c[l] += 1
                k = max(1, int(len(c) * 0.07)); sets.append(set(sorted(c, key=lambda l: -c[l])[:k]))
            self.repeat = float(np.mean([len(x & y) / max(len(x), 1) for x, y in zip(sets[:-1], sets[1:])]))
            stable = self.repeat < REPEAT_MIN
        self.stable = bool(stable) and bool(part_sizes) and len(part_sizes) > 1
        if self.stable:
            self.pid = []; 
            for i, n in enumerate(part_sizes): self.pid += [i] * n
            self.pid += [len(part_sizes) - 1] * (len(train) - len(self.pid)); self.np = len(part_sizes)

    def try_swap(self, hot, partner, moved, cur_len):
        saved, cost, w, al = super().try_swap(hot, partner, moved, cur_len)
        rf = getattr(self, "refill", None)
        if rf:   # refill trips: each product's refills now start from storage to its NEW face
            if rf[0] == "prod":   # measured: each product refills from its own storage origin
                _, rate, orow, ratio = rf; d = 0.0
                for a, b in ((hot, partner), (partner, hot)):
                    if a in orow: d += rate[a] * (orow[a][b] - orow[a][a])
                cost += 2 * self.W.dpu * ratio * d
            else:
                rate, row, ratio = rf
                cost += 2 * self.W.dpu * ratio * (rate[hot] * (row[partner] - row[hot]) + rate[partner] * (row[hot] - row[partner]))
        if self.stable:
            per = [0.0] * self.np
            for i, v in al.items(): per[self.pid[i]] += (cur_len[i] - v) * self.W.dpu
            saved = min(saved, self.np * min(per))
        return saved, cost, w, al
    def nearer_quiet(self, hot, moved):
        dh, vh = self.dep_dist[hot], self.vis.get(hot, 0)
        sc = []
        for l, v in self.vis.items():
            if l == hot or l in moved: continue
            dd, dv = dh - self.dep_dist[l], vh - v
            if dd > 0 and dv > 0: sc.append((dd * dv, l))
        sc.sort(reverse=True)
        return [l for _, l in sc]

    def plus(self, order, K=10, passes=3, ring=True):
        old = self.K; self.K = K
        moved, n, c = self.greedy(order, True)
        for _ in range(passes - 1 if ring else 0):
            m2, n2, c2 = self.greedy(order, True, partners=self.nearer_quiet, moved0=moved)
            if n2 == 0 and m2 == moved: break
            moved, n, c = m2, n + n2 - 0, c + c2
        self.K = old
        return moved, n, c

    def cluster_two_way(self, moved0, loc_cluster, order):
        vis_in = defaultdict(list)
        for l, c in loc_cluster.items(): vis_in[c].append(self.vis.get(l, 0))
        ranked = sorted(vis_in, key=lambda c: -np.mean(vis_in[c]))
        pool = sorted(loc_cluster, key=lambda l: self.dep_dist[l]); band, pos = {}, 0
        for c in ranked:
            for l in pool[pos:pos + len(vis_in[c])]: band[l] = c
            pos += len(vis_in[c])
        mis = [l for l in order if l in loc_cluster and band[l] != loc_cluster[l]]
        def match(hot, moved):   # partner sits in hot's target band, wears hot's current band label, so both land right
            tgt, here = loc_cluster[hot], band[hot]
            both = [l for l in loc_cluster if l not in moved and band[l] == tgt and loc_cluster[l] == here]
            one = [l for l in loc_cluster if l not in moved and band[l] == tgt and loc_cluster[l] != tgt and loc_cluster[l] != here]
            f = lambda l: (self.vis.get(l, 0), self.cases(l))
            return sorted(both, key=f) + sorted(one, key=f)
        return self.greedy(mis, True, partners=match, moved0=moved0)

    def rs_plus(self): return self.plus(self.order_detour, K=10, passes=3)


    # ---- Affinity v2: co-pick distance ------------------------------------------------------------
    def build_affinity(self, top=20, min_count=2):
        """Two locations are 'affine' when many tours visit both. Keep each location's strongest partners."""
        from itertools import combinations
        from collections import Counter
        cnt = Counter()
        for t in self.train:
            u = sorted(set(t))
            if len(u) > 40: u = u[:40]
            for a, b in combinations(u, 2): cnt[(a, b)] += 1
        nb = defaultdict(list)
        for (a, b), c in cnt.items():
            if c >= min_count: nb[a].append((c, b)); nb[b].append((c, a))
        self.aff = {a: sorted(v, reverse=True)[:top] for a, v in nb.items()}
        self.Dn = np.asarray(self.W.D)
        # pull = how much co-pick distance a location carries today: sum of (tours together x distance apart)
        self.pull = {a: sum(c * self.Dn[a][b] for c, b in v) for a, v in self.aff.items()}

    def aff_partners(self, hot, moved):
        """Quieter, visited locations ranked by how close they sit to the hot product's usual co-picked locations."""
        v = self.aff.get(hot)
        if not v: return []
        idx = np.array([b for _, b in v]); w = np.array([c for c, _ in v], float)
        score = self.Dn[:, idx] @ w                     # co-pick distance if the product lived in each location
        vh = self.vis.get(hot, 0)
        cand = [l for l in np.argsort(score)[:400] if l != hot and l not in moved and 0 < self.vis.get(l, 0) < vh]
        return [int(l) for l in cand]

    def affinity_v2(self, K=10, passes=2):
        self.build_affinity(); order = sorted(self.pull, key=lambda a: -self.pull[a])
        old = self.K; self.K = K; moved, n, c = self.greedy(order, True, partners=self.aff_partners)
        for _ in range(passes - 1):
            m2, n2, c2 = self.greedy(order, True, partners=self.aff_partners, moved0=moved)
            if m2 == moved: break
            moved, n, c = m2, n + n2, c + c2
        self.K = old; return moved, n, c


    # ---- Textbook family grouping (correlated storage assignment, no cost test) -------------------
    def family_grouping(self, n_swaps, min_jac=0.10, fam_size=8, cost_test=False):
        """Industry-standard family grouping: score location pairs by Jaccard (tours together / tours with either), group each busy
        location with its most similar unassigned locations, put the busiest families on the slots nearest the dispatch point.
        Swaps as many as ABC does (n_swaps), no money test."""
        from itertools import combinations
        from collections import Counter
        cnt = Counter()
        for t in self.train:
            u = sorted(set(t))[:40]
            for a, b in combinations(u, 2): cnt[(a, b)] += 1
        nb = defaultdict(list)
        for (a, b), c in cnt.items():
            j = c / (self.vis.get(a, 0) + self.vis.get(b, 0) - c)
            if j >= min_jac: nb[a].append((j, b)); nb[b].append((j, a))
        seen, fams = set(), []
        for s in sorted(self.vis, key=lambda l: -self.vis[l]):
            if s in seen: continue
            f = [s]; seen.add(s)
            for j, b in sorted(nb.get(s, []), reverse=True):
                if len(f) >= fam_size: break
                if b not in seen and b in self.vis: f.append(b); seen.add(b)
            fams.append(f)
        fams.sort(key=lambda f: -sum(self.vis[l] for l in f))
        slots = sorted(range(self.W.n), key=lambda l: self.dep_dist[l]); target, k = {}, 0
        for f in fams:
            for l in sorted(f, key=lambda l: -self.vis[l]):
                target[l] = slots[k]; k += 1
        order = [l for f in fams for l in sorted(f, key=lambda l: -self.vis[l])]
        parts = lambda hot, moved: [target[hot]] if target[hot] != hot and target[hot] not in moved else []
        old = self.K; moved, n, c = self.greedy(order, cost_test, n_force=n_swaps, partners=parts); self.K = old
        return moved, n, c


def limit_swaps(W, tours, moved, cap):
    """Operator limit: keep the first `cap` swaps in the order RackSmart accepted them (worst detour first,
    each one already past the cost test). cap None/0 or not binding = keep everything."""
    keep, seen = {}, set()
    for a, b in moved.items():
        if a in seen: continue
        if cap and len(seen) // 2 >= cap: break
        seen.add(a); seen.add(b); keep[a] = b; keep[b] = a
    return keep

def racksmart_sweep(inp, train, sizes, caps, window=1, demand_scale=1.0, cost_mult=1.0, guard=True, prep=None):
    """RackSmart: marginal detour picks WHO, co-pick distance picks WHERE, cost test, two passes, aisle fence,
    stability setting, then the operator's swap limit (first N accepted) and the backtest guard for each limit.
    caps: list of swap limits (None = no limit). One planning run serves every limit.
    Returns {cap: (moved, n_swaps, blocked, repeat)}. prep(pl) may add extra partners (e.g. clusters)."""
    Wp = World(inp.D, inp.depot, inp.cases, inp.golden, inp.dollars_per_unit * demand_scale / window, inp.cost_per_case * cost_mult)
    We = World(inp.D, inp.depot, inp.cases, inp.golden, inp.dollars_per_unit * demand_scale, inp.cost_per_case * cost_mult)
    faces = inp.extra.get("faces")
    grp = [1 if re.match(r"[A-Z]+", f).group(0) in ("GG", "HH") else 0 for f in faces] if faces else None   # aisle fence
    def plan(tr, szs):
        pl = PlannerPlus(Wp, tr, part_sizes=szs, stable="auto"); pl.build_affinity(); pl.K = 10
        if prep: prep(pl)
        order = [l for l in pl.order_detour if l in pl.aff]                       # WHO: ranked by marginal detour
        parts = (lambda hot, mv, _b=pl.aff_partners: [p for p in _b(hot, mv) if grp[p] == grp[hot]]) if grp else pl.aff_partners   # WHERE: co-pick distance
        moved, n, _ = pl.greedy(order, True, partners=parts)                      # pass 1, cost test inside
        m2, n2, _ = pl.greedy(order, True, partners=parts, moved0=moved)          # pass 2
        if m2 != moved: moved = m2
        return moved, pl.repeat
    moved, rep = plan(train, sizes)
    do_guard = guard and len(sizes) > 1
    if do_guard:
        k = sizes[-1]; older, latest = train[:-k], train[-k:]
        m_old, _ = plan(older, sizes[:-1])
    out = {}
    for cap in caps:
        mv = limit_swaps(Wp, train, moved, cap); blocked = 0
        if do_guard:                                                              # backtest guard, at this limit
            mo = limit_swaps(Wp, older, m_old, cap)
            s_old, _, _ = replay(We, latest, mo); c_old = move_cost(We, mo)
            proj = s_old * We.dpu * ((len(train) / window) / max(len(latest), 1)) - c_old
            if proj <= 0: mv, blocked = {}, 1
        out[cap] = (mv, len(mv) // 2, blocked, rep)
    return out

## 2. Load York data

In [ ]:
import os, re, numpy as np, pandas as pd
def load_york():
    cw = pd.read_csv(CROSSWALK); cw = cw[(cw.status == "ok") & cw.stop_id.notna() & (cw.level == "A")]
    sd = pd.read_csv(STOP_DIST, index_col=0)
    stops = list(sd.index); sidx = {s: i for i, s in enumerate(stops)}
    dock_stop = [s for s in stops if s.startswith("PD")][0]
    pk = pd.read_csv(PICKS, dtype={"ORD_NO": str}, low_memory=False)
    pk["dt"] = pd.to_datetime(pk.CREATE_DT, format="%m/%d/%Y %I:%M:%S %p", errors="coerce"); pk = pk.dropna(subset=["dt"])
    pk["loc"] = pk.FROM_LOCATION.astype(str).str.strip()
    stop_of = dict(zip(cw.location.str.strip(), cw.stop_id))
    n_all = len(pk); pk = pk[pk["loc"].isin(stop_of)]; print("picks mapped to a stop:", len(pk), "of", n_all)
    faces = sorted(pk["loc"].unique()); idx = {l: i for i, l in enumerate(faces)}; n = len(faces)
    print("pick-face locations:", n)
    si = np.array([sidx[stop_of[l]] for l in faces]); Ds = sd.values.astype(float)
    D = np.zeros((n + 1, n + 1)); D[:n, :n] = Ds[np.ix_(si, si)]
    D[:n, n] = Ds[si, sidx[dock_stop]]; D[n, :n] = Ds[sidx[dock_stop], si]
    inv = pd.read_csv(INVENTORY); inv = inv[inv.UOM == "CS"]
    cases_by = inv.assign(l=inv.LOCATION.astype(str).str.strip()).groupby("l").QTY.sum()
    typical = float(np.median(cases_by.values)); cases = [float(cases_by.get(l, typical)) for l in faces]   # missing = median location
    f = cw.set_index(cw.location.str.strip()).loc[faces]; ymax = f.groupby(f.stop_x_in.round(0)).stop_y_in.transform("max")
    golden = [int(i) for i in np.where(((ymax - f.stop_y_in).abs() / 12.0 <= 60).values)[0]]   # aisle-front rule (used by the ABC baselines)
    pk = pk.sort_values(["ORD_NO", "USER_NAME", "dt"]); pk["li"] = pk["loc"].map(idx); pk["month"] = pk.dt.dt.to_period("M").astype(str)
    by = {}
    for (o, u), g in pk.groupby(["ORD_NO", "USER_NAME"], sort=False):    # a tour = one operator's picks for one order (>= 3 stops)
        seq = [g.li.iloc[0]]
        for l in g.li.iloc[1:]:
            if l != seq[-1]: seq.append(l)
        if len(seq) >= 3: by.setdefault(g.month.iloc[0], []).append(seq)
    by = {m: v for m, v in by.items() if m not in (min(by), max(by))}      # drop the first and last (partial) months
    dpu = (WAGE / 3600.0) / (MPH * 5280 / 3600.0)                          # $ per foot walked
    print({m: len(v) for m, v in sorted(by.items())})
    loc_cluster = {}
    if CLUSTERS:   # location -> cluster of the product holding the most cases there (October inventory used as the product map for every month)
        cl = pd.read_csv(CLUSTERS); cof = dict(zip(cl.sku.astype(int), cl.cluster.astype(int)))
        top = inv.assign(l=inv.LOCATION.astype(str).str.strip()).sort_values("QTY", ascending=False).drop_duplicates("l").set_index("l").ITEM
        loc_cluster = {idx[l]: cof[int(i)] for l, i in top.items() if l in idx and int(i) in cof}
        print("locations with a cluster label:", len(loc_cluster), "of", n)
    return Inputs("york", D, n, cases, golden, dpu, COST_PER_CASE, by, unit_note="feet", extra=dict(faces=faces, loc_cluster=loc_cluster))
INP = load_york()

## 3. RackSmart planner

In [ ]:
def add_cluster_partners(pl, loc_cluster, max_mates=10):
    """SKU clusters as extra partners: a location's cluster mates (busiest first, among locations with training picks) join its co-pick partner list
    with the median weight of its own partners, then the pull scores are recomputed."""
    if not loc_cluster: return
    mates = {}
    for l, c in loc_cluster.items():
        if pl.vis.get(l, 0) > 0: mates.setdefault(c, []).append(l)
    for a in list(pl.aff):
        if a not in loc_cluster: continue
        base = pl.aff[a]; w = max(2, int(np.median([c for c, _ in base]))) if base else 2; have = {b for _, b in base}
        ms = sorted([m for m in mates.get(loc_cluster[a], []) if m != a and m not in have], key=lambda m: -pl.vis.get(m, 0))[:max_mates]
        pl.aff[a] = sorted(base + [(w, m) for m in ms], reverse=True)
    pl.pull = {a: sum(c * pl.Dn[a][b] for c, b in v) for a, v in pl.aff.items()}
REPEAT_MIN_NOTE = "stability setting switches on below 60% repeat (REPEAT_MIN in the engine)"
def racksmart(inp, train, sizes, window=1, demand_scale=1.0, cost_mult=1.0, guard=True, max_swaps=-1):
    """Returns (moved, n_swaps, blocked, repeat). moved = {location: location it swaps with}.
    WHO = marginal detour, WHERE = co-pick distance (+ clusters), cost test, two passes, aisle fence, stability setting,
    then the operator's limit (first MAX_SWAPS accepted) and the backtest guard. The engine is racksmart_sweep (in the engine code above)."""
    cap = MAX_SWAPS if max_swaps == -1 else max_swaps
    prep = lambda pl: add_cluster_partners(pl, inp.extra.get("loc_cluster"))      # SKU clusters as extra partners
    return racksmart_sweep(inp, train, sizes, [cap], window, demand_scale, cost_mult, guard, prep)[cap]
def swap_list(inp, moved):
    faces = inp.extra["faces"]; seen = set(); rows = []
    for a, b in moved.items():
        if (b, a) in seen: continue
        seen.add((a, b)); rows.append(dict(location_a=faces[a], location_b=faces[b], cases_a=inp.cases[a], cases_b=inp.cases[b]))
    return pd.DataFrame(rows)

## 4. Comparison (plan on the previous month, replay on the next month's tours)

In [ ]:
def run_month(inp, te, demand_scale=1.0, cost_mult=1.0):
    P = sorted(inp.tours_by_period); k = P.index(te); tr = P[k - 1]
    train, test = inp.tours_by_period[tr], inp.tours_by_period[te]; sizes = [len(train) // 2, len(train) - len(train) // 2]
    Wp = World(inp.D, inp.depot, inp.cases, inp.golden, inp.dollars_per_unit * demand_scale, inp.cost_per_case * cost_mult)
    We = Wp; out = []
    def score(name, moved, n, extra=None):
        s, t, p = replay(We, test, moved); c = move_cost(We, moved)
        out.append(dict(month=te, method=name, swaps=n, walking_cut_pct=round(100 * p, 1), gross=round(s * We.dpu), move_cost=round(c), net=round(s * We.dpu - c)))
    pl = PlannerPlus(Wp, train, part_sizes=sizes, stable=False)
    score("Do nothing", {}, 0)
    m, n_det, _ = pl.plan("detour+cost")
    m, n, _ = pl.plan("classic_abc", n_ref=n_det); score("Classic ABC (no cost test)", m, n)
    m, n, _ = pl.plan("velocity+cost"); score("ABC with cost test", m, n)
    m, n, _ = pl.family_grouping(n_det, cost_test=True); score("Family grouping + cost test", m, n)
    moved, n, blocked, rep = racksmart(inp, train, sizes, demand_scale=demand_scale, cost_mult=cost_mult)
    score("RackSmart", moved, n)
    return out, moved
months = sorted(INP.tours_by_period)[1:]
months = TEST_MONTHS or months[-4:]
rows = []; plans = {}
for te in months:
    r, mv = run_month(INP, te, DEMAND_SCALE, COST_MULT); rows += r; plans[te] = mv; print("done", te, flush=True)
res = pd.DataFrame(rows)
summary = res.groupby("method", sort=False)[["swaps", "walking_cut_pct", "gross", "move_cost", "net"]].mean().round(1)
print(f"\nYork | demand x{DEMAND_SCALE} | moving cost x{COST_MULT} | test months {months} | mean per month")
summary

In [ ]:
# Save results and the swap list for the last test month
res.to_csv("racksmart_results_by_month.csv", index=False); summary.to_csv("racksmart_results_summary.csv")
sl = swap_list(INP, plans[months[-1]]); sl.to_csv("racksmart_swap_list_%s.csv" % months[-1], index=False)
print(len(sl), "swaps in the plan for", months[-1]); sl.head(10)

## Notes and limitations
- **Distances.** Walking distances come from the travel network (feet). Only level A pick faces are included. About 2% of picks occur in aisles RR and ZZ, which are not yet represented on the network.
- **Inventory.** Cases per location come from a single inventory snapshot, applied to every month, and are used only for the cost of moving stock. Locations without an inventory record use the median.
- **Not modeled.** Pallet-to-case conversion, refill trips, rack height rules, congestion in aisles, and product-specific handling rules. Only the GG/HH aisle fence is enforced.
- **Test months.** By default the last four months are run; set `TEST_MONTHS` to choose others (ten months, November 2025 to August 2026, are available). Differences of a few percent between variants cannot be resolved with this few months.
- **Interim controls.** The stability setting and the backtest guard are interim measures. They are expected to be revisited once planning is driven by the demand forecast.
- **Clusters.** Clusters label about 21% of York locations and are attached to locations through the October inventory, which assumes the same product occupied each face in earlier months. In testing they changed results by about 1%.
- **Swap limit.** `MAX_SWAPS` limits the amount of work a plan asks of the crew; it does not increase the saving. The first swaps carry most of the value. The planner also enforces an internal ceiling of 150 swaps per pass.
- **Second warehouse.** The same engine runs on the public footwear dataset through a separate loader, which is not included in this notebook.